# KaKaRoTiOS — Dragon Ball Z: Kakarot

O notebook reutiliza **o mesmo DRIVE_FOLDER_ID já configurado pelo Link2iOS/SpkIOSzero**. Dentro dessa pasta compartilhada existe a subpasta `kakarot`. Os 20 DLCs estão diretamente nela, cada um com seu próprio nome: 17 arquivos com aproximadamente 124 KB e 3 arquivos com aproximadamente 106 KB. O `prod.keys` pode ficar na raiz configurada ou dentro de `kakarot/`.

Os seis endereços são **links de compartilhamento AkiraBox**: BASE (`010051C0134F8000`, `v0`) e UPDATE (`010051C0134F8800`, `v1048576`), três partes por conjunto. Antes de qualquer transferência, o notebook consulta a API oficial de status e confere o nome de cada arquivo, Title ID, versão e ordem `part1/part2/part3`. Depois tenta abrir o botão normal de download com uma sessão HTTP comum. Não contorna CAPTCHA/Cloudflare: se a plataforma exigir verificação humana, o processo para com uma mensagem clara, sem repetir dez vezes um `403`.

Base, update, os 20 DLCs, RomFS, AOC e workspace permanecem separados. Os RARs só são apagados depois da verificação dos pacotes reconstruídos no Drive. O despacho para GitHub permanece desligado por padrão.

In [ ]:
# Links de compartilhamento AkiraBox: validados via API oficial antes do primeiro download.
DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"

BASE_PART_URLS = [
    "https://akirabox.to/1RgzR95eo3bp/file",  # BASE part1
    "https://akirabox.to/rk9zKWobZz0l/file",  # BASE part2
    "https://akirabox.to/E1x3AWPwnGyk/file",  # BASE part3
]
UPDATE_PART_URLS = [
    "https://akirabox.to/2bJG8bw273OB/file",  # UPDATE part1
    "https://akirabox.to/dkD3YwyYAGLQ/file",  # UPDATE part2
    "https://akirabox.to/jar3Xl78j32d/file",  # UPDATE part3
]

# Nomes locais lógicos: o mesmo prefixo é necessário para o UnRAR encadear as partes.
# O prefixo aleatório AkiraBox varia por URL; estes nomes normalizam apenas os arquivos locais.
BASE_PART_NAMES = [
    "Dragon Ball Z Kakarot [010051C0134F8000][v0][taodung.com].part1.rar",
    "Dragon Ball Z Kakarot [010051C0134F8000][v0][taodung.com].part2.rar",
    "Dragon Ball Z Kakarot [010051C0134F8000][v0][taodung.com].part3.rar",
]
UPDATE_PART_NAMES = [
    "Dragon Ball Z Kakarot [010051C0134F8800][v1048576][taodung.com].part1.rar",
    "Dragon Ball Z Kakarot [010051C0134F8800][v1048576][taodung.com].part2.rar",
    "Dragon Ball Z Kakarot [010051C0134F8800][v1048576][taodung.com].part3.rar",
]

# Mantido desligado: nenhum GitHub Actions será disparado nesta preparação.
DISPATCH_TO_GITHUB = False

In [ ]:
from pathlib import Path
import argparse, json, os, subprocess, sys
from google.colab import auth

auth.authenticate_user()
ROOT = Path("/content/kakarot-ios-repo")
if not ROOT.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "KaKaRoTiOS",
        "https://github.com/joao9vegeto-stack/KintTany.git", str(ROOT)
    ], check=True)
else:
    branch = subprocess.check_output(
        ["git", "-C", str(ROOT), "branch", "--show-current"], text=True
    ).strip()
    if branch != "KaKaRoTiOS":
        raise RuntimeError(f"Workspace existente está na branch {branch!r}; não vou trocar de branch.")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", "KaKaRoTiOS"], check=True)
    subprocess.run(["git", "-C", str(ROOT), "reset", "--hard", "origin/KaKaRoTiOS"], check=True)

assert subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip() == "KaKaRoTiOS"
sys.path.insert(0, str(ROOT))
PY_FILES = [
    "colab/source_kakarot.py", "colab/prepare_kakarot.py", "colab/run_kakarot.py",
    "scripts/prepare_kakarot_actions.py", "scripts/build_kakarot_shared.py",
]
subprocess.run([sys.executable, "-m", "py_compile", *[str(ROOT / item) for item in PY_FILES]], check=True)
print("KaKaRoTiOS Python syntax: OK")
print("Repository:", ROOT)
print("Branch:", "KaKaRoTiOS")

In [ ]:
from colab.run_kakarot import prepare, recover
from colab.source_kakarot import preflight_akirabox_links
from urllib.parse import unquote, urlparse
import re

def assert_rar_group(urls, expected_title_id, expected_version, label):
    if len(urls) != 3 or len(set(urls)) != 3:
        raise ValueError(f"{label}: são exigidos três links distintos.")
    for number, url in enumerate(urls, 1):
        parsed = urlparse(url)
        host = (parsed.hostname or "").lower()
        path = unquote(parsed.path)
        if host in ("akirabox.to", "www.akirabox.to", "akirabox.com", "www.akirabox.com"):
            if not re.fullmatch(r"/[A-Za-z0-9_-]+/file/?", path):
                raise ValueError(f"{label}: link {number} não tem formato AkiraBox /<id>/file.")
        else:
            path_lower = path.lower()
            if not path_lower.endswith(f".part{number}.rar"):
                raise ValueError(f"{label}: link {number} não é o volume part{number}.rar.")
            if f"[{expected_title_id.lower()}]" not in path_lower or f"[{expected_version.lower()}]" not in path_lower:
                raise ValueError(f"{label}: ID/versão inesperado no link {number}; interrompendo.")

if all(x.strip() for x in BASE_PART_URLS + UPDATE_PART_URLS):
    assert_rar_group(BASE_PART_URLS, "010051C0134F8000", "v0", "BASE")
    assert_rar_group(UPDATE_PART_URLS, "010051C0134F8800", "v1048576", "UPDATE")
    preflight_akirabox_links([
        {"label": "BASE", "urls": BASE_PART_URLS, "names": BASE_PART_NAMES,
         "title_id": "010051C0134F8000", "version": "v0"},
        {"label": "UPDATE", "urls": UPDATE_PART_URLS, "names": UPDATE_PART_NAMES,
         "title_id": "010051C0134F8800", "version": "v1048576"},
    ])
    handoff = prepare(argparse.Namespace(
        drive_folder_id=DRIVE_FOLDER_ID,
        base_part_urls=[x.strip() for x in BASE_PART_URLS],
        update_part_urls=[x.strip() for x in UPDATE_PART_URLS],
        base_part_names=BASE_PART_NAMES,
        update_part_names=UPDATE_PART_NAMES,
    ))
else:
    print("Os seis links ainda não estão preenchidos. Tentando recuperar uma preparação anterior do Drive...")
    handoff = recover(argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID))
    if handoff is None:
        raise ValueError("Preencha os três BASE_PART_URLS e os três UPDATE_PART_URLS, na ordem volume 1, 2, 3.")

print("\nPreparação recuperada/concluída:", handoff["request_id"])
print("Drive:", handoff["drive_output_folder_url"])


In [ ]:
# Auditoria antes de qualquer despacho para GitHub Actions.
prepared = handoff["prepared"]
print(json.dumps({
    "project": handoff["project"],
    "branch": handoff["branch"],
    "title_id": handoff["title_id"],
    "package_version": handoff["package_version"],
    "base_drive": handoff["game_data"]["base_package"],
    "update_drive": handoff["game_data"]["update_package"],
    "rar_parts_deleted": handoff["source"]["rar_parts_deleted_after_upload"],
    "profile_type": prepared["profile_type"],
    "profile_source": prepared["profile_source"],
    "main_build_id": prepared["main_build_id"],
    "main_sha256": prepared["main_sha256"],
    "modules": prepared["modules"],
    "graphics_api": prepared["graphics_api"],
    "unreal_shader_sources": prepared["unreal_shader_sources"],
    "pak_audit": prepared["pak_audit"],
    "shader_chain": prepared["shader_chain"],
    "romfs_size": prepared["romfs_size"],
    "romfs_sha256": prepared["romfs_sha256"],
    "aoc_count": prepared["aoc_count"],
    "aoc_images": prepared["aoc_images"],
    "drive_workspace": handoff["work_archive"],
}, indent=2))

In [ ]:
# DESPACHO EXPLÍCITO — só execute depois de revisar o handoff acima.
if DISPATCH_TO_GITHUB:
    from colab.run_kakarot import dispatch
    dispatch(argparse.Namespace(handoff=Path("/content/kakarot-ios/handoff.json")))
else:
    print("Actions NÃO disparada. Para enviar, altere DISPATCH_TO_GITHUB = True e execute esta célula.")